# 03 — Brain Tumor Detection (presence + bounding box)

Detection sits between classification and segmentation:
- **Classification** → "is there a tumor?"
- **Detection**     → "is there a tumor, AND where is it (bbox)?"
- **Segmentation**  → "which pixels are tumor?"

Lecture p.92 mentions the three main types of object-recognition algorithms. Here we build the simplest one: a **single-object detector** with two heads on a CNN backbone — `presence` (binary) and `bbox` (cx, cy, w, h).

**Bounding-box source.** The LGG MRI dataset gives us masks; we derive a bbox per slice as the smallest rectangle containing the mask (see `data._mask_to_bbox`). Slices without a tumor get `presence = 0` and an ignored bbox.

**Audience note (AD researchers).** Detection-style heads show up in *lesion localization* (cerebral microbleeds, lacunes, hyperintensities). The same encoder + (presence, bbox) head pattern applies — just swap the dataset.

## 0. Setup

```bash
pip install torch torchvision matplotlib numpy pillow kagglehub
```

In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

import sys
from pathlib import Path

PROJ_DIR = Path.cwd().parent
sys.path.insert(0, str(PROJ_DIR))

import torch
import numpy as np
import matplotlib.pyplot as plt

from src.models import SimpleDetector, count_parameters
from src.data import get_device, get_detection_dataloaders
from src.train import train_detector, evaluate_detector
from src.viz import plot_history, plot_detection_samples, plot_detection_predictions

torch.manual_seed(42); np.random.seed(42)
device = get_device()
print(f"Using device: {device}")

## 1. Load the detection data

Same patient-level split as the segmentation notebook. Bboxes are computed on the fly from the masks.

In [ ]:
IMG_SIZE = 128
BATCH_SIZE = 32

train_loader, val_loader, test_loader, (train_ds, val_ds, test_ds) = get_detection_dataloaders(
    data_dir=str(PROJ_DIR / "data"),
    batch_size=BATCH_SIZE,
    img_size=IMG_SIZE,
    augment=True,
)

X, pres, bbox = next(iter(train_loader))
print(f"image    : {tuple(X.shape)}")
print(f"presence : {tuple(pres.shape)}   (0=no tumor, 1=tumor)")
print(f"bbox     : {tuple(bbox.shape)}   (cx, cy, w, h)")
print(f"\npresence in batch: {pres.tolist()}")
print(f"bbox example (pos): {bbox[pres > 0.5][0].tolist() if (pres>0.5).any() else 'no tumor in batch'}")

In [ ]:
# Visualize a few ground-truth bboxes
_ = plot_detection_samples(train_ds, n=6, title="Ground-truth bboxes (derived from masks)")
plt.show()

## 2. Define the detector

Backbone: 4 conv-BN-ReLU-pool blocks → AdaptiveAvgPool → 256-D feature vector.
Two heads on top:
- `presence_head`: Linear(256 → 1), BCE loss
- `bbox_head`: Linear(256 → 4) + sigmoid, Smooth-L1 loss (only on positive samples)

This is intentionally minimal — the goal is to teach the *idea* of multi-head outputs that share a backbone.

In [ ]:
model = SimpleDetector(in_channels=1)
print(f"Trainable params: {count_parameters(model):,}")

dummy = torch.randn(2, 1, IMG_SIZE, IMG_SIZE)
p_logits, bb = model(dummy)
print(f"presence_logits: {tuple(p_logits.shape)}   bbox: {tuple(bb.shape)}")

## 3. Train

Loss = `BCE(presence) + 5 * SmoothL1(bbox)` — the 5x weight is a typical Faster-RCNN-style choice to balance classification and regression scales. Bbox loss is masked to positives only.

In [ ]:
EPOCHS = 10
LR = 1e-3
WD = 1e-4

history = train_detector(
    model, train_loader, val_loader,
    epochs=EPOCHS, lr=LR, weight_decay=WD, bbox_weight=5.0, device=device,
)

In [ ]:
# Plot loss and presence-accuracy / IoU together
fig, ax = plt.subplots(1, 3, figsize=(14, 4))
epochs = range(1, EPOCHS + 1)
ax[0].plot(epochs, history['train_loss'], 'o-', label='train')
ax[0].plot(epochs, history['val_loss'], 's-', label='val')
ax[0].set_xlabel('Epoch'); ax[0].set_ylabel('Total loss'); ax[0].set_title('Loss')
ax[0].legend(); ax[0].grid(alpha=0.3)

ax[1].plot(epochs, [a*100 for a in history['val_presence_acc']], 'o-', color='tab:green')
ax[1].set_xlabel('Epoch'); ax[1].set_ylabel('Val presence accuracy (%)')
ax[1].set_title('Presence classification'); ax[1].grid(alpha=0.3)

ax[2].plot(epochs, history['val_mean_iou_pos'], 'o-', color='tab:red')
ax[2].set_xlabel('Epoch'); ax[2].set_ylabel('Mean IoU (positives only)')
ax[2].set_title('Bbox quality'); ax[2].grid(alpha=0.3)

plt.tight_layout(); plt.show()

## 4. Test-set evaluation

We report two numbers:
- **presence accuracy**: did we get "tumor / no tumor" right?
- **mean IoU (positives)**: among slices that actually have a tumor, how well does the predicted bbox overlap the ground-truth bbox?

In [ ]:
test_loss, test_pres_acc, test_miou = evaluate_detector(model, test_loader, device)
print(
    f"[TEST] loss={test_loss:.4f} "
    f"presence_acc={test_pres_acc*100:.2f}% "
    f"mean_iou(positives)={test_miou:.4f}"
)

In [ ]:
# Green = ground truth, red = prediction
_ = plot_detection_predictions(model, test_ds, n=6, device=device, threshold=0.5)
plt.show()

## 5. Save

In [ ]:
out_dir = PROJ_DIR / "outputs"
out_dir.mkdir(exist_ok=True)
torch.save({
    "model_state": model.state_dict(),
    "history": history,
    "test_pres_acc": test_pres_acc,
    "test_miou": test_miou,
}, out_dir / "detect_simple.pt")
print("Saved.")

## 6. Hands-on exercises

### Architecture
1. **Decoupled heads.** Add an MLP (Linear-ReLU-Linear) before each of `presence_head` and `bbox_head`. Does decoupling help?
2. **Anchor-free vs. anchor-based.** Our model predicts one bbox per image (no anchors, no NMS). What changes if you want to detect *two* tumors per slice?
3. **IoU loss.** Replace `SmoothL1Loss` on bboxes with a differentiable IoU/GIoU loss.

### Loss balance
4. **Weight sweep.** Try `bbox_weight ∈ {0, 1, 5, 20}` in `train_detector`. What is the trade-off between presence accuracy and bbox IoU?
5. **Hard negatives.** Most slices have no tumor — the model can score high on presence by always predicting 0. Compute presence sensitivity vs. specificity separately. How would a class-weighted BCE change behavior?

### Going further
6. **torchvision FasterRCNN.** Re-wire the detection dataset to return torchvision's expected format (`{'boxes': tensor, 'labels': tensor}`) and fine-tune `fasterrcnn_resnet50_fpn`. Compare IoU.
7. **Detection ↔ segmentation handoff.** Use the bbox to crop, then run the U-Net from notebook 02 on the cropped region. Does this *two-stage* approach beat global segmentation?

### Translate to AD
8. **Microbleed detection.** Cerebral microbleeds on SWI are tiny, sparse, multi-instance findings. Sketch how you'd extend `SimpleDetector` to output a *list* of bboxes per scan (hint: grid of anchors + per-cell heads, like YOLO).